In [1]:
%run ../bootstrap.py

[Bootstrap] cwd=/home/ikk/renal-calculi-classificaction | seed=57


In [2]:
import json
from pathlib import Path
from typing import Optional

import pandas as pd


METRICS_TO_SHOW = [
    "accuracy",
    "f1_weighted",
    "precision_weighted",
    "recall_weighted",
    "auc_weighted",
]
METRIC_NAMES_MAP = {
    "accuracy": "Accuracy",
    "f1_weighted": "F1",
    "precision_weighted": "Precision",
    "recall_weighted": "Recall",
    "auc_weighted": "AUC",
}


def load_summary(summary_path: Path) -> dict:
    with open(summary_path) as f:
        return json.load(f)


def format_mean_std(
    summary: dict,
    metrics: list[str],
    decimals: int = 0,
) -> dict[str, str]:
    formatted = {}
    for metric in metrics:
        mean = summary.get(f"{metric}_mean")
        std = summary.get(f"{metric}_std")
        if mean is None or std is None:
            formatted[metric] = "—"
            continue
        mean_pct = mean * 100
        std_pct = std * 100
        formatted[metric] = f"{mean_pct:.{decimals}f}±{std_pct:.{decimals}f}"
    return formatted


def find_summary_file(
    experiment_dir: Path,
    level: str = "default",
) -> Path:
    
    if level == "image":
        path = experiment_dir / "per_image_summary.json"
        if not path.exists():
            raise FileNotFoundError(
                f"No existe per_image_summary.json en {experiment_dir}"
            )
        return path
    
    for candidate in ["summary.json", "kfold_summary.json"]:
        path = experiment_dir / candidate
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No se encontró ni summary.json ni kfold_summary.json en {experiment_dir}"
    )


def build_metrics_table(
    path_root: Path,
    models: list[str],
    task: str,
    caso: str,
    level: str = "default",
    metrics: list[str] = METRICS_TO_SHOW,
    suffix: str = "",
) -> pd.DataFrame:
    """
    Construye una tabla con columnas = metrics y filas = models, 
    para una sola tarea (binario o multiclase).
    
    Estructura de carpetas esperada:
        path_root/{model}/{model}_{caso}/{model}_{task}{suffix}/
    
    Parámetros
    ----------
    path_root : raíz de los experimentos.
    models : lista de modelos.
    task : 'binario' o 'multiclase'.
    caso : 'images' o 'patches'.
    level : 'default' o 'image'.
    metrics : lista de métricas a incluir.
    suffix : sufijo opcional añadido al final del nombre del experimento. 
             Por ejemplo "_with_augmentation" o "_no_augmentation". 
             Si se deja vacío, se usa el nombre base {model}_{task}.
    """
    rows = {}
    for model in models:
        experiment_dir = (
            path_root / model / f"{model}_{caso}" / f"{model}_{task}{suffix}"
        )
        try:
            summary_path = find_summary_file(experiment_dir, level=level)
            summary = load_summary(summary_path)
            formatted = format_mean_std(summary, metrics)
        except FileNotFoundError as e:
            print(f"[AVISO] {e}")
            formatted = {m: "—" for m in metrics}
        rows[model] = formatted
    
    df = pd.DataFrame.from_dict(rows, orient="index")
    df = df[metrics]
    df.rename(columns=METRIC_NAMES_MAP, inplace=True)
    df.index.name = "model"
    return df


def build_summary_tables(
    caso: str,
    path_root: Path = Path("experiments/"),
    models: list[str] = None,
    tasks: list[str] = None,
    metrics: list[str] = None,
    suffix: str = "",
) -> dict[str, pd.DataFrame]:
    """
    Construye las tablas resumen del experimento, separadas por tarea.
    
    Parámetros
    ----------
    caso : "images" o "patches".
        - "images":  una tabla por tarea (binario, multiclase).
        - "patches": dos tablas por tarea (nivel patch y nivel imagen).
    suffix : sufijo opcional añadido al nombre de la carpeta del experimento. 
             Por ejemplo "_with_augmentation". Se pasa a build_metrics_table.
    
    Devuelve
    --------
    dict {nombre_tabla: DataFrame}
    """
    if caso not in ("images", "patches"):
        raise ValueError(f"caso debe ser 'images' o 'patches', recibido: {caso}")
    
    if models is None:
        models = ["xgboost", "resnet50", "hybrid_resnet50"]
    if tasks is None:
        tasks = ["binario", "multiclase"]
    if metrics is None:
        metrics = METRICS_TO_SHOW
    
    tables = {}
    table_suffix = suffix  # se añade también al nombre de la tabla
    
    if caso == "images":
        for task in tasks:
            tables[f"images_{task}{table_suffix}"] = build_metrics_table(
                path_root, models, task,
                caso="images",
                level="default",
                metrics=metrics,
                suffix=suffix,
            )
    else:  # patches
        for task in tasks:
            tables[f"patches_patch_level_{task}{table_suffix}"] = build_metrics_table(
                path_root, models, task,
                caso="patches",
                level="default",
                metrics=metrics,
                suffix=suffix,
            )
            tables[f"patches_image_level_{task}{table_suffix}"] = build_metrics_table(
                path_root, models, task,
                caso="patches",
                level="image",
                metrics=metrics,
                suffix=suffix,
            )
    
    return tables


def save_summary_tables(
    tables: dict[str, pd.DataFrame],
    output_dir: Optional[Path] = None,
    print_to_console: bool = True,
):
    """Imprime las tablas y opcionalmente las guarda en CSV y LaTeX."""
    titles = {
        "images_binario":              "Clasificación binaria con imágenes",
        "images_multiclase":           "Clasificación multiclase con imágenes",
        "patches_patch_level_binario": "Clasificación binaria con patches — nivel patch",
        "patches_patch_level_multiclase":
                                       "Clasificación multiclase con patches — nivel patch",
        "patches_image_level_binario":
                                       "Clasificación binaria con patches — nivel imagen",
        "patches_image_level_multiclase":
                                       "Clasificación multiclase con patches — nivel imagen",
    }
    
    for name, df in tables.items():
        # Para el título, eliminar el sufijo si no está en el mapa
        title = titles.get(name)
        if title is None:
            # Buscar título base sin sufijo (para variantes augmented, etc.)
            for base_name, base_title in titles.items():
                if name.startswith(base_name):
                    extra = name[len(base_name):]
                    title = f"{base_title} ({extra.lstrip('_').replace('_', ' ')})"
                    break
            if title is None:
                title = name
        
        if print_to_console:
            print(f"\n{title}")
            print(df.to_string())
        
        if output_dir is not None:
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)
            df.to_csv(output_dir / f"summary_{name}.csv")
            df.to_latex(
                output_dir / f"summary_{name}.tex",
                escape=False,
                bold_rows=True,
            )

In [86]:
without= build_summary_tables(caso="patches", suffix= "_no_augmentation",  tasks= ["multiclase"])
save_summary_tables(without, output_dir=Path("experiments/summary_tables"))


Clasificación multiclase con patches — nivel patch (no augmentation)
                Accuracy    F1 Precision Recall   AUC
model                                                
xgboost             66±4  65±3      68±2   66±4  85±3
resnet50            69±2  67±2      68±2   69±2  87±1
hybrid_resnet50     69±2  68±2      70±2   69±2  88±1

Clasificación multiclase con patches — nivel imagen (soft voting) (no augmentation)
                Accuracy    F1 Precision Recall   AUC
model                                                
xgboost             71±5  69±5      70±6   71±5  89±4
resnet50            72±4  69±5      70±5   72±4  91±2
hybrid_resnet50     74±5  71±6      73±6   74±5  92±2


In [80]:
witht= build_summary_tables(caso="patches", suffix= "_with_augmentation",  tasks= ["multiclase"])
save_summary_tables(witht, output_dir=Path("experiments/summary_tables"))



Clasificación multiclase con patches — nivel patch (with augmentation)
                Accuracy    F1 Precision Recall   AUC
model                                                
xgboost             55±4  55±5      58±4   55±4  83±2
resnet50            62±5  62±5      64±3   62±5  88±3
hybrid_resnet50     63±5  63±5      65±4   63±5  89±3

Clasificación multiclase con patches — nivel imagen (soft voting) (with augmentation)
                Accuracy    F1 Precision Recall   AUC
model                                                
xgboost             69±3  70±2      73±3   69±3  89±1
resnet50            77±3  76±3      78±3   77±3  95±1
hybrid_resnet50     77±2  77±2      79±2   77±2  95±1


In [10]:
import json
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd


DEFAULT_MODELS = ["xgboost", "resnet50", "hybrid_resnet50"]
DEFAULT_METRICS = ["precision", "recall", "f1"]


def compute_per_class_metrics_summary(
    fold_results_path: str,
    class_names: Optional[list[str]] = None,
    metrics: Optional[list[str]] = None,
    decimals: int = 0,
    include_support: bool = True,
) -> pd.DataFrame:
    """
    Calcula media ± std entre folds de las métricas por clase para un único 
    modelo, leyendo el JSON de fold_results.
    """
    with open(fold_results_path) as f:
        fold_results = json.load(f)
    
    first_per_class = fold_results[0].get("per_class", {})
    if class_names is None:
        class_names = first_per_class.get("class_names")
        if class_names is None:
            n_classes = len(first_per_class.get("f1", []))
            class_names = [f"class_{i}" for i in range(n_classes)]
    
    if metrics is None:
        metrics = DEFAULT_METRICS
    
    n_folds = len(fold_results)
    n_classes = len(class_names)
    
    metric_arrays = {m: np.zeros((n_folds, n_classes)) for m in metrics}
    for i, fr in enumerate(fold_results):
        per_class = fr["per_class"]
        for m in metrics:
            values = per_class.get(m)
            if values is None:
                raise KeyError(
                    f"La métrica '{m}' no está en per_class del fold {i}. "
                    f"Disponibles: {list(per_class.keys())}"
                )
            metric_arrays[m][i, :] = values
    
    rows = {}
    for c_idx, cname in enumerate(class_names):
        row = {}
        for m in metrics:
            mean = metric_arrays[m][:, c_idx].mean() * 100
            std = metric_arrays[m][:, c_idx].std() * 100
            row[m] = f"{mean:.{decimals}f} ± {std:.{decimals}f}"
        
        if include_support:
            support_per_fold = [
                fr["classification_report"][cname]["support"]
                for fr in fold_results
                if cname in fr.get("classification_report", {})
            ]
            if support_per_fold:
                row["support"] = int(sum(support_per_fold))
        rows[cname] = row
    
    df = pd.DataFrame.from_dict(rows, orient="index")
    df.index.name = "class"
    return df


def build_per_class_combined_table(
    task: str,
    case: str,
    path_root: Path = Path("experiments/"),
    models: Optional[list[str]] = None,
    metrics: Optional[list[str]] = None,
    decimals: int = 0,
    suffix: str = "",
) -> pd.DataFrame:
    """
    Construye una tabla combinada de métricas por clase para los tres modelos.
    
    Parámetros
    ----------
    task : 'binario' o 'multiclase'.
    case : 'images' o 'patches'.
    path_root : Raíz de los experimentos.
    models : Lista de modelos. Por defecto los tres canónicos.
    metrics : Métricas por clase. Por defecto ['precision', 'recall', 'f1'].
    decimals : Decimales para el formato 'mean ± std'.
    suffix : Sufijo opcional añadido al nombre de la carpeta del experimento.
    
    Devuelve
    --------
    DataFrame con MultiIndex de columnas (modelo, métrica), filas = clases, 
    valores en formato 'mean ± std'. La última columna es el soporte total 
    (común a todos los modelos).
    
    Estructura de carpetas esperada:
        path_root/{model}/{model}_{case}/{model}_{task}{suffix}/fold_results.json
    """
    if task not in ("binario", "multiclase"):
        raise ValueError(f"task debe ser 'binario' o 'multiclase', recibido: {task}")
    if case not in ("images", "patches"):
        raise ValueError(f"case debe ser 'images' o 'patches', recibido: {case}")
    
    if models is None:
        models = DEFAULT_MODELS
    if metrics is None:
        metrics = DEFAULT_METRICS
    
    per_model_dfs = {}
    support_series = None
    
    for model in models:
        experiment_dir = (
            path_root / model / f"{model}_{case}" / f"{model}_{task}{suffix}"
        )
        print(f"experiments_dir = {experiment_dir}")
        fold_results_path = experiment_dir / "fold_results.json"
        
        if not fold_results_path.exists():
            print(f"[AVISO] No existe: {fold_results_path}")
            continue
        
        df_model = compute_per_class_metrics_summary(
            fold_results_path=str(fold_results_path),
            metrics=metrics,
            decimals=decimals,
            include_support=True,
        )
        
        # Guardar el soporte una sola vez (debe ser el mismo para todos los modelos)
        if "support" in df_model.columns:
            if support_series is None:
                support_series = df_model["support"]
            df_model = df_model.drop(columns=["support"])
        
        per_model_dfs[model] = df_model
    
    if not per_model_dfs:
        raise FileNotFoundError(
            f"No se encontró ningún fold_results.json para task={task}, case={case}"
        )
    
    # Concatenar todos los modelos en una sola tabla con MultiIndex de columnas
    combined = pd.concat(per_model_dfs, axis=1)
    combined.columns.names = ["model", "metric"]
    
    # Añadir la columna de soporte al final
    if support_series is not None:
        combined[("", "support")] = support_series
    
    return combined


def save_per_class_combined_table(
    df: pd.DataFrame,
    output_dir: Path,
    task: str,
    case: str,
    suffix: str = "",
    print_to_console: bool = True,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    name = f"per_class_{case}_{task}{suffix}"
    
    if print_to_console:
        print(f"\nMétricas por clase — {case} / {task}{suffix}")
        print(df.to_string())
    
    df.to_latex(
        output_dir / f"{name}.tex",
        escape=False,
        multicolumn_format="c",
        bold_rows=True,
    )
    print(f"\n[INFO] Guardado en {output_dir / name}.csv y .tex")

In [11]:
df = build_per_class_combined_table(
    task="multiclase",
    # models= ["xgboost"],
    case="patches",
    suffix= "_no_augmentation"
)


experiments_dir = experiments/xgboost/xgboost_patches/xgboost_multiclase_no_augmentation
experiments_dir = experiments/resnet50/resnet50_patches/resnet50_multiclase_no_augmentation
experiments_dir = experiments/hybrid_resnet50/hybrid_resnet50_patches/hybrid_resnet50_multiclase_no_augmentation


In [5]:
save_per_class_combined_table(
    df,
    output_dir=Path("experiments/summary_tables"),
    task="multiclase",
    case="paches",
    # suffix= "_using_smote"
)


Métricas por clase — paches / multiclase
model        xgboost                    resnet50                   hybrid_resnet50                          
metric     precision   recall       f1 precision   recall       f1       precision   recall       f1 support
class                                                                                                       
COM           75 ± 3   84 ± 6   79 ± 4    76 ± 3   89 ± 4   82 ± 2          76 ± 4   89 ± 4   82 ± 3    2767
COD          45 ± 17  31 ± 25   30 ± 8   40 ± 15   22 ± 8   25 ± 4         44 ± 11   20 ± 5   27 ± 5     298
Mixto        51 ± 15   52 ± 2   51 ± 9    54 ± 8   51 ± 5   52 ± 4          55 ± 9   53 ± 6   53 ± 3    1372
Infeccioso   60 ± 19  50 ± 15   50 ± 8   62 ± 18  53 ± 13   54 ± 8         59 ± 21  54 ± 15   51 ± 6     485
Urico        60 ± 14  57 ± 19  55 ± 11   65 ± 12  61 ± 14  61 ± 10          74 ± 9  61 ± 16  65 ± 10     368

[INFO] Guardado en experiments/summary_tables/per_class_paches_multiclase.csv y .tex
